In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import mutual_info_score, accuracy_score
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression

url = 'https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/course_lead_scoring_2026.csv'
df = pd.read_csv(url)
df.head()

,lead_source,industry,employment_status,location,annual_income,number_of_courses_viewed,interaction_count,lead_score,converted
0,organic_search,technology,employed,europe,88160.0,3,4,0.64,1
1,social_media,technology,employed,south_america,72688.0,3,7,0.72,1
2,referral,retail,employed,europe,44697.0,3,4,0.58,1
3,paid_ads,manufacturing,student,NaN,NaN,3,6,0.57,0
4,referral,manufacturing,student,north_america,24062.0,4,5,0.62,1


In [2]:
print(df.isnull().sum())
df.dtypes

lead_source                 148
industry                    240
employment_status           194
location                    208
annual_income               369
number_of_courses_viewed      0
interaction_count             0
lead_score                   35
converted                     0
dtype: int64


lead_source                     str
industry                        str
employment_status               str
location                        str
annual_income               float64
number_of_courses_viewed      int64
interaction_count             int64
lead_score                  float64
converted                     int64
dtype: object

In [3]:
categorical = df.select_dtypes(include='object').columns.tolist()
numerical = [c for c in df.select_dtypes(exclude='object').columns if c != 'converted']
print(categorical, numerical)

df[categorical] = df[categorical].fillna('NA')
df[numerical] = df[numerical].fillna(0.0)
df.isnull().sum().sum()

['lead_source', 'industry', 'employment_status', 'location'] ['annual_income', 'number_of_courses_viewed', 'interaction_count', 'lead_score']


C:\Users\nirum\AppData\Local\Temp\ipykernel_10592\2926954998.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical = df.select_dtypes(include='object').columns.tolist()


np.int64(0)

In [4]:
df['industry'].mode()[0]

'technology'

In [5]:
corr = df[numerical].corr()
display(corr)

pairs = [
    ('interaction_count', 'lead_score'),
    ('number_of_courses_viewed', 'lead_score'),
    ('number_of_courses_viewed', 'interaction_count'),
    ('annual_income', 'interaction_count'),
]
for a, b in pairs:
    print(f'{a:28s} {b:20s} {corr.loc[a, b]:.4f}')

,annual_income,number_of_courses_viewed,interaction_count,lead_score
annual_income,1.000000,0.161300,0.122842,0.229496
number_of_courses_viewed,0.161300,1.000000,0.721609,0.757204
interaction_count,0.122842,0.721609,1.000000,0.915746
lead_score,0.229496,0.757204,0.915746,1.000000


interaction_count            lead_score           0.9157
number_of_courses_viewed     lead_score           0.7572
number_of_courses_viewed     interaction_count    0.7216
annual_income                interaction_count    0.1228


In [6]:
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=42)
df_train, df_val = train_test_split(df_full_train, test_size=0.25, random_state=42)

y_train = df_train['converted'].values
y_val = df_val['converted'].values
y_test = df_test['converted'].values

df_train = df_train.drop(columns='converted').reset_index(drop=True)
df_val = df_val.drop(columns='converted').reset_index(drop=True)
df_test = df_test.drop(columns='converted').reset_index(drop=True)
len(df_train), len(df_val), len(df_test)

(3000, 1000, 1000)

In [7]:
df_train_mi = df_train.copy()
df_train_mi['converted'] = y_train

mi = {c: round(mutual_info_score(df_train_mi[c], df_train_mi['converted']), 2) for c in categorical}
pd.Series(mi).sort_values(ascending=False)

lead_source          0.03
employment_status    0.02
industry             0.00
location             0.00
dtype: float64

In [8]:
def train_model(df_tr, y_tr, cat, num, C=1.0):
    dv = DictVectorizer(sparse=False)
    X = dv.fit_transform(df_tr[cat + num].to_dict(orient='records'))
    model = LogisticRegression(solver='liblinear', C=C, max_iter=1000, random_state=42)
    model.fit(X, y_tr)
    return dv, model

def evaluate(dv, model, df_v, y_v, cat, num):
    X = dv.transform(df_v[cat + num].to_dict(orient='records'))
    return accuracy_score(y_v, model.predict(X))

features_cat, features_num = categorical, numerical
dv, model = train_model(df_train, y_train, features_cat, features_num)
base_acc = evaluate(dv, model, df_val, y_val, features_cat, features_num)
print('accuracy:', round(base_acc, 2), '(unrounded:', base_acc, ')')

accuracy: 0.65 (unrounded: 0.645 )


In [9]:
all_features = features_cat + features_num
rows = []
for f in all_features:
    cat = [c for c in features_cat if c != f]
    num = [c for c in features_num if c != f]
    dv_f, m_f = train_model(df_train, y_train, cat, num)
    acc = evaluate(dv_f, m_f, df_val, y_val, cat, num)
    rows.append((f, acc, base_acc - acc))

res = pd.DataFrame(rows, columns=['feature', 'accuracy_without', 'difference']).sort_values('difference')
res

,feature,accuracy_without,difference
4,annual_income,0.724,-0.079
1,industry,0.645,0.000
3,location,0.645,0.000
2,employment_status,0.645,0.000
7,lead_score,0.644,0.001
5,number_of_courses_viewed,0.643,0.002
0,lead_source,0.642,0.003
6,interaction_count,0.601,0.044


In [10]:
candidates = ['lead_source', 'number_of_courses_viewed', 'interaction_count']
res[res['feature'].isin(candidates)]

,feature,accuracy_without,difference
5,number_of_courses_viewed,0.643,0.002
0,lead_source,0.642,0.003
6,interaction_count,0.601,0.044


In [11]:
for C in [0.000001, 0.00001, 0.0001, 0.001]:
    dv_c, m_c = train_model(df_train, y_train, features_cat, features_num, C=C)
    acc = evaluate(dv_c, m_c, df_val, y_val, features_cat, features_num)
    print(f'C={C:<10} accuracy={round(acc, 3)}')

C=1e-06      accuracy=0.598
C=1e-05      accuracy=0.598
C=0.0001     accuracy=0.613
C=0.001      accuracy=0.645
